In [ ]:
import pandas as pd
import psycopg2
import DATABASE_CONFIG

conn = psycopg2.connect(
    dbname=DATABASE_CONFIG.DB_NAME,
    user=DATABASE_CONFIG.DB_USER,
    password=DATABASE_CONFIG.DB_PASSWORD,
    host=DATABASE_CONFIG.DB_HOST,
    port=DATABASE_CONFIG.DB_PORT
)

cursor = conn.cursor()

In [ ]:
import re

df = pd.read_csv('../datasets/DesmatamentoAreasIndigena.csv', sep=';')
df = df[['indi']]
df = df.rename(columns={
    'indi': 'nome',
    })

for index, row in df.iterrows():
    nome = row['nome']
    if 'restrição' in nome:
        nome = re.sub(r'\s*\(.*?restrição.*?\)', '', nome, flags=re.IGNORECASE)
        df.at[index, 'nome'] = nome

data = list(df.itertuples(index=False, name=None))

In [12]:
from psycopg2.extras import execute_values
comando = """
    INSERT INTO area_indigena 
    (nome)
    VALUES %s
    ON CONFLICT (nome) DO NOTHING;
"""
execute_values(cursor, comando, data)

conn.commit()


In [ ]:
conn.rollback()

In [13]:
cursor.close()
conn.close()